# CPS 535: Introduction to Large Language Models
## Week 1: Neural Network Fundamentals

**By the end of this week you will be able to:**
- Explain what a neuron computes and why we chain them into layers
- Explain why activation functions must be non-linear, and what ReLU/Sigmoid do differently
- Compute a full forward pass, loss, and backward pass **by hand** for a tiny network
- Implement that exact same network manually in numpy, then in PyTorch
- Extend the network to multiple inputs and train it on a small dataset in PyTorch


## 1. The Neuron

A neuron takes one or more inputs, computes a **weighted sum plus a bias**, and passes the result through an **activation function**.

For a single input $x$, weight $w$, and bias $b$:

$$z = w \cdot x + b$$
$$a = f(z)$$

- $z$ is called the **pre-activation** (or "logit") which is just a linear combination of inputs.
- $a$ is the **activation** which is the neuron's actual output, after applying a non-linear function $f$.
- $w$ and $b$ are the **learnable parameters** that are the numbers that training will adjust.

We can stack many neurons into a **layer**, and stack layers to get a **network**. The input is also a layer called the **input layer**. The first layer(s) are called **hidden layers**; the final layer is the **output layer**.


## 2. Activation Functions

**If every layer only computed a weighted sum with no activation function, stacking layers would be pointless.** A linear function of a linear function is still just a linear function:

$$z_2 = w_2(w_1 x + b_1) + b_2 = (w_2 w_1)x + (w_2 b_1 + b_2)$$

No matter how many layers you stack, the whole network collapses to a single linear equation. A single neuron could only draw a straight line. **Non-linear activation functions are what let a network represent curves, bends, and complex decision boundaries.**

Two activation functions we'll use constantly this course:

**ReLU** (Rectified Linear Unit) used in hidden layers:
$$\text{ReLU}(z) = \max(0, z)$$
Simple, fast, and avoids a problem called "vanishing gradients" that older activations (like sigmoid) suffer from in deep networks. It's the default choice for hidden layers in modern networks, including transformers.

**Sigmoid** used in output layers for binary classification:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
Squashes any real number into the range $(0, 1)$, so the output can be interpreted as a probability. This is exactly what we want when predicting "probability the label is 1."


## 3. Loss Functions

A loss function measures how wrong the network's prediction $\hat{y}$ is compared to the true label $y$. Training is the process of adjusting weights and biases to make the loss as small as possible.

For binary classification (label is 0 or 1, prediction is a probability), we use **Binary Cross-Entropy (BCE)**:

$$L = -\big[y \log(\hat{y}) + (1-y)\log(1-\hat{y})\big]$$

Intuition: if $y=1$, the loss simplifies to $-\log(\hat{y})$ which means it's small when $\hat y$ is close to 1 (confident and correct) and grows sharply as $\hat y \to 0$ (confident and wrong). This sharp penalty for confident-wrong predictions is exactly what we want during classification training.


## 4. Forward Pass and Backward Pass

**Forward pass:** feed the input through the network, layer by layer, to produce a prediction and compute the loss. This is just repeated application of $z = wx+b$, $a=f(z)$.

**Backward pass (backpropagation):** to reduce the loss, we need to know *how much each weight and bias contributed to the error* i.e., the gradient of the loss with respect to every parameter, $\frac{\partial L}{\partial w}$. Backpropagation computes these gradients efficiently by applying the **chain rule** backward through the network, layer by layer, reusing intermediate results instead of recomputing everything from scratch.

**Gradient descent:** once we have every gradient, we nudge each parameter a small step in the direction that reduces the loss:

$$w \leftarrow w - \eta \frac{\partial L}{\partial w} \qquad b \leftarrow b - \eta \frac{\partial L}{\partial b}$$

where $\eta$ (eta) is the **learning rate** which tells how big a step to take. Repeating forward pass → loss → backward pass → update, many times, is the entire training loop that every neural network including every LLM uses.


## 5. Example

Let's compute every one of these steps by hand for a tiny concrete network, so nothing is a black box before we write a single line of code.

**Network setup**
- Input: $x = 2$
- Hidden layer (1 neuron): weight $w_1 = 0.8$, bias $b_1 = 0$, activation = ReLU → produces $z_1$, $a_1$
- Output layer (1 neuron): weight $w_2 = 0.8$, bias $b_2 = 0.2$, activation = Sigmoid → produces $z_2$, $\hat{y}$
- True label: $y = 1$

Architecture: $x \rightarrow [w_1, b_1] \rightarrow z_1 \rightarrow \text{ReLU} \rightarrow a_1 \rightarrow [w_2, b_2] \rightarrow z_2 \rightarrow \text{Sigmoid} \rightarrow \hat y$


### 5.1 Forward Pass (Hidden Layer)

$$z_1 = w_1 \cdot x + b_1 = 0.8 \times 2 + 0 = 1.6$$

$$a_1 = \text{ReLU}(z_1) = \max(0, 1.6) = 1.6$$

(Since $z_1 > 0$, ReLU passes it through unchanged.)


### 5.2 Forward Pass (Output Layer)

$$z_2 = w_2 \cdot a_1 + b_2 = 0.8 \times 1.6 + 0.2 = 1.28 + 0.2 = 1.48$$

$$\hat{y} = \sigma(z_2) = \frac{1}{1+e^{-1.48}} = \frac{1}{1 + 0.2276} \approx 0.8146$$

Our network currently predicts $\hat y \approx 0.8146$, a fairly confident (though not yet trained) probability that the label is 1.


### 5.3 Loss

True label $y = 1$, so BCE simplifies to $-\log(\hat y)$:

$$L = -\log(0.8146) \approx 0.2051$$


### 5.4 Backward Pass

We need $\frac{\partial L}{\partial w_1}, \frac{\partial L}{\partial b_1}, \frac{\partial L}{\partial w_2}, \frac{\partial L}{\partial b_2}$. Work backward from the loss, one layer at a time.

**Step 1 (output layer):** For sigmoid output + BCE loss, the combined derivative simplifies neatly to:

$$\frac{\partial L}{\partial z_2} = \hat y - y = 0.8146 - 1 = -0.1854$$

Then, since $z_2 = w_2 a_1 + b_2$:

$$\frac{\partial L}{\partial w_2} = \frac{\partial L}{\partial z_2} \cdot a_1 = -0.1854 \times 1.6 \approx -0.2967$$

$$\frac{\partial L}{\partial b_2} = \frac{\partial L}{\partial z_2} \cdot 1 = -0.1854$$

**Step 2 (propagate into the hidden layer):** First, how much did $a_1$ affect the loss (through $z_2 = w_2 a_1 + b_2$)?

$$\frac{\partial L}{\partial a_1} = \frac{\partial L}{\partial z_2} \cdot w_2 = -0.1854 \times 0.8 \approx -0.1483$$

Then pass through ReLU's derivative. $\text{ReLU}'(z_1) = 1$ if $z_1 > 0$, else $0$. Since $z_1 = 1.6 > 0$:

$$\frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial a_1} \cdot \text{ReLU}'(z_1) = -0.1483 \times 1 = -0.1483$$

Finally, since $z_1 = w_1 x + b_1$:

$$\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial z_1} \cdot x = -0.1483 \times 2 \approx -0.2967$$

$$\frac{\partial L}{\partial b_1} = \frac{\partial L}{\partial z_1} \cdot 1 = -0.1483$$

This backward chain output error → scaled by downstream weight → passed through the activation's derivative → scaled by upstream input. This is backpropagation. In a real network, this same pattern repeats through every layer, and PyTorch's `autograd` automates exactly this bookkeeping.


### 5.5 Gradient Descent Update

Using learning rate $\eta = 0.1$:

$$w_2 \leftarrow 0.8 - 0.1 \times (-0.2967) = 0.8297$$
$$b_2 \leftarrow 0.2 - 0.1 \times (-0.1854) = 0.2185$$
$$w_1 \leftarrow 0.8 - 0.1 \times (-0.2967) = 0.8297$$
$$b_1 \leftarrow 0 - 0.1 \times (-0.1483) = 0.0148$$

All four parameters moved in the direction that would have made $\hat y$ closer to $y=1$ — exactly what we want, since our (untrained, random-in-general) starting weights happened to already push the prediction toward the correct class. In a real network these updates repeat for thousands of steps across a full dataset, not just one example.


## 6. Manual Implementation (numpy)

Now let's implement the exact same network in code and confirm every number matches what we computed by hand above.

In [1]:
import numpy as np

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return float(z > 0)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x = 2.0
w1, b1 = 0.8, 0.0
w2, b2 = 0.8, 0.2
y_true = 1.0
lr = 0.1

In [2]:
# --- Forward pass ---
z1 = w1 * x + b1
a1 = relu(z1)

z2 = w2 * a1 + b2
y_pred = sigmoid(z2)

print(f"z1 = {z1:.4f}")
print(f"a1 = {a1:.4f}")
print(f"z2 = {z2:.4f}")
print(f"y_pred = {y_pred:.4f}")

z1 = 1.6000
a1 = 1.6000
z2 = 1.4800
y_pred = 0.8146


In [3]:
# --- Loss ---
eps = 1e-8  # avoid log(0)
loss = -(y_true * np.log(y_pred + eps) + (1 - y_true) * np.log(1 - y_pred + eps))
print(f"Loss = {loss:.4f}")

Loss = 0.2051


In [4]:
# --- Backward pass ---
dz2 = y_pred - y_true                  # combined sigmoid + BCE derivative
dw2 = dz2 * a1
db2 = dz2

da1 = dz2 * w2
dz1 = da1 * relu_derivative(z1)
dw1 = dz1 * x
db1 = dz1

print(f"dL/dw2 = {dw2:.4f}")
print(f"dL/db2 = {db2:.4f}")
print(f"dL/dw1 = {dw1:.4f}")
print(f"dL/db1 = {db1:.4f}")

dL/dw2 = -0.2967
dL/db2 = -0.1854
dL/dw1 = -0.2967
dL/db1 = -0.1483


In [10]:
# --- Gradient descent update ---
w2_new = w2 - lr * dw2
b2_new = b2 - lr * db2
w1_new = w1 - lr * dw1
b1_new = b1 - lr * db1

print(f"w2: {w2:.4f} -> {w2_new:.4f}")
print(f"b2: {b2:.4f} -> {b2_new:.4f}")
print(f"w1: {w1:.4f} -> {w1_new:.4f}")
print(f"b1: {b1:.4f} -> {b1_new:.4f}")

print("\nCompare these to Section 5.5; they should match.")

w2: 0.8000 -> 0.8297
b2: 0.2000 -> 0.2185
w1: 0.8000 -> 0.8297
b1: 0.0000 -> 0.0148

Compare these to Section 5.5; they should match.


**🔧 Try it:** Change `y_true` to `0.0` and re-run all cells above. Recompute the loss and gradients by hand on paper first, then verify your numbers against the code output.

## 7. Same Example, in PyTorch

We just derived and coded gradients by hand. PyTorch's `autograd` computes them automatically. Now, let's confirm it produces identical numbers for the exact same network.

In [6]:
import torch

# requires_grad=True tells PyTorch to track operations for gradient computation
w1_t = torch.tensor(0.8, requires_grad=True)
b1_t = torch.tensor(0.0, requires_grad=True)
w2_t = torch.tensor(0.8, requires_grad=True)
b2_t = torch.tensor(0.2, requires_grad=True)

x_t = torch.tensor(2.0)
y_true_t = torch.tensor(1.0)

# --- Forward pass ---
z1_t = w1_t * x_t + b1_t
a1_t = torch.relu(z1_t)

z2_t = w2_t * a1_t + b2_t
y_pred_t = torch.sigmoid(z2_t)

print(f"z1 = {z1_t.item():.4f}")
print(f"a1 = {a1_t.item():.4f}")
print(f"z2 = {z2_t.item():.4f}")
print(f"y_pred = {y_pred_t.item():.4f}")

z1 = 1.6000
a1 = 1.6000
z2 = 1.4800
y_pred = 0.8146


In [7]:
# --- Loss ---
loss_t = torch.nn.functional.binary_cross_entropy(y_pred_t, y_true_t)
print(f"Loss = {loss_t.item():.4f}")

# Backward pass: one line instead of manual chain-rule
loss_t.backward()

print(f"\ndL/dw2 = {w2_t.grad.item():.4f}")
print(f"dL/db2 = {b2_t.grad.item():.4f}")
print(f"dL/dw1 = {w1_t.grad.item():.4f}")
print(f"dL/db1 = {b1_t.grad.item():.4f}")
print("\nThese should match Section 6's manual gradients exactly.")

Loss = 0.2051

dL/dw2 = -0.2967
dL/db2 = -0.1854
dL/dw1 = -0.2967
dL/db1 = -0.1483

These should match Section 6's manual gradients exactly.


In [11]:
# --- Gradient descent update (manual, to show it's the same math) ---
with torch.no_grad():
    w2_t -= lr * w2_t.grad
    b2_t -= lr * b2_t.grad
    w1_t -= lr * w1_t.grad
    b1_t -= lr * b1_t.grad

print(f"w2: -> {w2_t.item():.4f}")
print(f"b2: -> {b2_t.item():.4f}")
print(f"w1: -> {w1_t.item():.4f}")
print(f"b1: -> {b1_t.item():.4f}")
print("\nCompare to Section 6 and Section 5.5; all three approaches should agree.")

w2: -> 0.8593
b2: -> 0.2371
w1: -> 0.8593
b1: -> 0.0297

Compare to Section 6 and Section 5.5; all three approaches should agree.


In practice, we don't manually update each parameter or zero gradients by hand, instead we use `torch.nn.Module` for the network and `torch.optim` for the update step, exactly as in last lab's Week 1 NN Fundamentals notebook. The manual version above exists purely so you can see that `loss.backward()` is doing precisely the chain-rule computation from Section 5.4, not something mysterious.

## 8. Extending the Example (Two Inputs, in PyTorch)

Real inputs are rarely a single number. Let's extend our tiny network to accept **two** input features, using proper PyTorch modules this time (`nn.Linear`, `nn.Module`) instead of individual scalar tensors.

With two inputs $x_1, x_2$, the hidden neuron's pre-activation becomes:

$$z_1 = w_{1a} x_1 + w_{1b} x_2 + b_1$$

Everything else (ReLU, output layer, sigmoid, BCE loss, backprop, gradient descent) works exactly the same (the extra input just means one extra weight to learn).

In [9]:
import torch.nn as nn

class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 1)   # 2 inputs -> 1 hidden neuron
        self.output = nn.Linear(1, 1)   # 1 hidden neuron -> 1 output

    def forward(self, x):
        z1 = self.hidden(x)
        a1 = torch.relu(z1)
        z2 = self.output(a1)
        y_pred = torch.sigmoid(z2)
        return y_pred

torch.manual_seed(0)
model = TinyNet()

# A single example with 2 input features
x_example = torch.tensor([[2.0, 1.0]])
y_true_example = torch.tensor([[1.0]])

y_pred_example = model(x_example)
print(f"y_pred = {y_pred_example.item():.4f}")

loss_fn = nn.BCELoss()
loss_example = loss_fn(y_pred_example, y_true_example)
print(f"Loss = {loss_example.item():.4f}")

loss_example.backward()
print("\nGradients now exist for every weight and bias in the network:")
for name, param in model.named_parameters():
    print(f"  {name}: grad = {param.grad}")

y_pred = 0.4049
Loss = 0.9042

Gradients now exist for every weight and bias in the network:
  hidden.weight: grad = tensor([[0., 0.]])
  hidden.bias: grad = tensor([0.])
  output.weight: grad = tensor([[0.]])
  output.bias: grad = tensor([-0.5951])


Notice the weights are randomly initialized (via `nn.Linear`'s default init) rather than the fixed `0.8` we used by hand. In practice, we never hand-pick starting weights; training finds good values from a random start, as you saw in last lab's full training loops.

## 9. Exercise (Train `TinyNet` on a Small Dataset)

Now it's your turn. Below is a small synthetic 2-feature binary classification dataset. Extend `TinyNet` above (or reuse it) and write a full training loop.

**Requirements:**
1. Use `nn.BCELoss()` as your loss function
2. Use `torch.optim.Adam` as your optimizer (recall from last lab why Adam is generally a strong default)
3. Train for at least 200 epochs, tracking the loss every epoch
4. Plot the loss curve (it should trend downward)
5. Report the final training accuracy (fraction of predictions, rounded to 0 or 1, that match `y`)

Some starter code and structure is provided. Your job is to fill in the `# TODO` sections.

In [ ]:
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt

X, y = make_classification(
    n_samples=200, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, random_state=7
)
y = y.reshape(-1, 1)

X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap="coolwarm", edgecolors="k")
plt.title("Exercise dataset")
plt.show()

In [ ]:
# TODO: instantiate your model (you can reuse TinyNet, or make the hidden layer wider)
model = None  # TODO

# TODO: define your loss function
loss_fn = None  # TODO

# TODO: define your optimizer (use Adam)
optimizer = None  # TODO

epochs = 200
losses = []

for epoch in range(epochs):
    # TODO: zero the gradients
    # TODO: forward pass -> y_pred
    # TODO: compute loss
    # TODO: backward pass
    # TODO: optimizer step
    # TODO: record the loss (losses.append(...))
    pass

# TODO: plot the loss curve

# TODO: compute and print final training accuracy


## References:
- Deep Learning by Ian Goodfellow and Yoshua Bengio and Aaron Courville ([Link](https://www.deeplearningbook.org/))
- Basics of PyTorch ([Link](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial2/Introduction_to_PyTorch.html))
